<a href="https://colab.research.google.com/github/TeemGambit/Generative-AI-D01-Classwork/blob/main/GenAI/HW4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

I was unable to successfully run the code.
I apologize for the incompleteness.

In [1]:
!pip install -qU langchain-core langchain-community langchain-google-genai langchain-classic
!pip install -qU faiss-cpu pypdf rank_bm25 flashrank

# Standard imports for environment management
import os
from google.colab import userdata

# API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.2/504.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.1 MB/s eta 0:00:00


SecretNotFoundError: Secret GOOGLE_API_KEY does not exist.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# Updated list to match your specific filenames
file_names = [
    "Cooking Basics.pdf",
    "Cooking Basics for ummies.pdf",
    "Thanksgiving from Scratch.pdf"
]

all_docs = []

# Loop through each file, load its content, and split it into chunks
for file in file_names:
    if os.path.exists(file):
        print(f"Processing {file}...")
        loader = PyPDFLoader(file)

        # Recursive splitter attempts to keep paragraphs and sentences together
        # chunk_size is total characters; overlap keeps context across boundaries
        splitter = RecursiveCharacterTextSplitter(chunk_size=850, chunk_overlap=80)
        chunks = loader.load_and_split(splitter)

        all_docs.extend(chunks)
    else:
        print(f"Warning: '{file}' not found in the sidebar. Please upload it.")

if all_docs:
    print(f"\nSuccess! Total text chunks created: {len(all_docs)}")

In [ ]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma



# Initialize the 2026 flagship embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT",
    persist_directory="./chroma_db" # Added for consistency
)



# Initialize Chroma (using the modular 2026 syntax)
vectorstore = Chroma(
    collection_name="digitate_report_2026",
    embedding_function=embeddings
)


# Clear any old data
ids = vectorstore.get()['ids']
if ids:
    vectorstore.delete(ids=ids)
    print(f"Cleared {len(ids)} old entries from the vectorstore.")



# Batch upload with rate-limit protection (Mandatory for Free Tier)
batch_size = 50
print(f" Embedding {len(splits)} chunks...")

for i in range(0, len(splits), batch_size):
    batch = splits[i : i + batch_size]
    vectorstore.add_documents(batch)
    print(f" Indexed chunks {i} to {min(i + batch_size, len(splits))}")
    if i + batch_size < len(splits):
        time.sleep(15)



# --- RETRIEVER TEST SECTION ---

# Convert the vectorstore into a retriever
# k=4 remains the 2026 standard for high-density reports
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})



# Execute a test query
query = "How do I make soup taste better?"
results = retriever.invoke(query)




# Print the results with enhanced Metadata (source tracking) formatting
print(f"\n Testing Retriever with query: '{query}'")

for i, doc in enumerate(results):
    # PyPDFLoader (pypdf) defaults to 0-indexed 'page'
    page_num = doc.metadata.get('page', 'Unknown')
    display_page = page_num + 1 if isinstance(page_num, int) else "Unknown"

    print(f"\n--- Result {i+1} | Source: Page {display_page} ---")

    # CLEANING: Replace multiple newlines/tabs with a single space for readability
    clean_content = " ".join(doc.page_content.split())

    # SNIPPET: Showing 350 chars (slightly more for 2026 technical analysis)
    print(f"Content: {clean_content[:350]}...")

    # OPTIONAL: Print the specific source file if metadata exists
    source_file = doc.metadata.get('source', 'Unknown File')
    if source_file != 'Unknown File':
        print(f"File: {source_file.split('/')[-1]}")

In [ ]:
import time
from tenacity import retry, stop_after_attempt, wait_random_exponential, retry_if_exception_type
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# Re-initialize Gemini Embeddings (2026 Stable Model)
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)



# Define the Retry-Safe Embedding Wrapper
# This function will wait and retry automatically if we get a 429 error.
@retry(
    wait=wait_random_exponential(min=1, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_exception_type(Exception) # Catch the GoogleGenerativeAIError
)
def embed_with_retry(vector_store, batch):
    if vector_store is None:
        return FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)
        return vector_store



# Process with Batching and Retries
batch_size = 15 # Smaller batches help prevent hitting the limit too fast
vectorstore = None

print(f"Embedding {len(all_docs)} chunks with rate-limit protection...")
for i in range(0, len(all_docs), batch_size):
    batch = all_docs[i : i + batch_size]
    try:
        vectorstore = embed_with_retry(vectorstore, batch)
        print(f" Processed {i + len(batch)}/{len(all_docs)}...")
    except Exception as e:
        print(f" Failed after retries: {e}")
        break
    time.sleep(3) # A steady 3-second heartbeat to keep the API happy




# Finalize Retrievers
if vectorstore:
    vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    bm25_retriever = BM25Retriever.from_documents(all_docs)
    bm25_retriever.k = 5

    hybrid_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )
    print("\n Hybrid Retriever is ready!")

